In [19]:
# Cell 1: Environment Setup
!pip install pandas numpy scikit-learn matplotlib pyarrow -q

import pandas as pd
import numpy as np
import os, json, csv, time, random, logging
from datetime import date, datetime, timedelta

random.seed(42)
np.random.seed(42)
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
print("✓ Environment ready for: Structured vs Semi-Structured Data")

✓ Environment ready for: Structured vs Semi-Structured Data



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\9901359\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [20]:
# Cell 2: Generate Dataset
PRODUCTS = ['Laptop_with_extended_description','Monitor_with_extended_description','Keyboard_with_extended_description','Mouse_with_extended_description','Headset_with_extended_description','Webcam_with_extended_description','USB Hub_with_extended_description','Desk Lamp_with_extended_description','Speaker_with_extended_description','Tablet_with_extended_description']
REGIONS = ['North_with_extended_description','South_with_extended_description','East_with_extended_description','West_with_extended_description']
STATUSES = ['completed','pending','cancelled','refunded']

records = []
for i in range(10000):
    records.append({
        'order_id': f'ORD-{i:05d}',
        'product': random.choice(PRODUCTS),
        'category': PRODUCTS[i % len(PRODUCTS)].split()[0] if i < 50 else random.choice(['Electronics','Accessories','Home','Sports','Office']),
        'region': random.choice(REGIONS),
        'quantity': random.randint(1, 50),
        'unit_price': round(random.uniform(9.99, 999.99), 2),
        'order_date': str(date(2024,1,1) + timedelta(days=random.randint(0,364))),
        'status': random.choices(STATUSES, weights=[70,15,10,5])[0]
    })

df = pd.DataFrame(records)
df['revenue'] = (df['quantity'] * df['unit_price']).round(2)

print(f"✓ Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

✓ Dataset loaded: 10000 rows, 9 columns


,order_id,product,category,region,quantity,unit_price,order_date,status,revenue
0,ORD-00000,Monitor_with_extended_description,Laptop_with_extended_description,North_with_extended_description,48,282.27,2024-04-24,completed,13548.96
1,ORD-00001,Monitor_with_extended_description,Monitor_with_extended_description,North_with_extended_description,38,427.69,2024-01-16,completed,16252.22
2,ORD-00002,Mouse_with_extended_description,Keyboard_with_extended_description,North_with_extended_description,36,206.84,2024-11-28,pending,7446.24
3,ORD-00003,USB Hub_with_extended_description,Mouse_with_extended_description,South_with_extended_description,29,593.36,2024-01-04,pending,17207.44
4,ORD-00004,Keyboard_with_extended_description,Headset_with_extended_description,West_with_extended_description,22,285.08,2024-04-20,refunded,6271.76


In [21]:
# Cell 3: Data Profiling
print('=' * 60)
print('DATA PROFILE: Structured vs Semi-Structured Data')
print('=' * 60)
print(f'\nShape: {df.shape}')
print(f'\nColumn Types:\n{df.dtypes}')
print(f'\nNull Counts:\n{df.isnull().sum()}')
print(f'\nDuplicate Rows: {df.duplicated().sum()}')
print(f'\nNumeric Summary:\n{df.describe()}')
print(f'\nUnique Values:')
for col in df.columns:
    print(f'  {col}: {df[col].nunique()} unique')
print(f'\nMemory Usage: {df.memory_usage(deep=True).sum() / 1024:.1f} KB')

DATA PROFILE: Structured vs Semi-Structured Data

Shape: (10000, 9)

Column Types:
order_id          str
product           str
category          str
region            str
quantity        int64
unit_price    float64
order_date        str
status            str
revenue       float64
dtype: object

Null Counts:
order_id      0
product       0
category      0
region        0
quantity      0
unit_price    0
order_date    0
status        0
revenue       0
dtype: int64

Duplicate Rows: 0

Numeric Summary:
           quantity    unit_price       revenue
count  10000.000000  10000.000000  10000.000000
mean      25.140600    503.807532  12629.256019
std       14.580209    287.654131  11033.664281
min        1.000000     10.050000     16.440000
25%       12.000000    254.082500   3529.290000
50%       25.000000    502.315000   9347.180000
75%       38.000000    751.495000  19358.000000
max       50.000000    999.970000  49685.500000

Unique Values:
  order_id: 10000 unique
  product: 10 unique
  c

In [22]:
# Cell 4: Core Implementation — Structured vs Semi-Structured Data

import csv, json, random, os, time
from datetime import date, timedelta
import pyarrow as pa
import pyarrow.parquet as pq

# Reproducible synthetic data generation
random.seed(42)
PRODUCTS = ['Laptop_with_extended_description','Monitor_with_extended_description','Keyboard_with_extended_description','Mouse_with_extended_description','Headset_with_extended_description','Webcam_with_extended_description','USB Hub_with_extended_description','Desk Lamp_with_extended_description','Speaker_with_extended_description','Tablet_with_extended_description']
REGIONS = ['North_with_extended_description','South_with_extended_description','East_with_extended_description','West_with_extended_description']

def make_record(i):
    return {
        'order_id': f'ORD-{i:05d}',
        'product': random.choice(PRODUCTS),
        'region': random.choice(REGIONS),
        'quantity': random.randint(1, 50),
        'unit_price': round(random.uniform(9.99, 999.99), 2),
        'order_date': str(date(2024, 1, 1) + timedelta(days=random.randint(0, 364))),
    }

records = [make_record(i) for i in range(100)]

# Save as CSV
os.makedirs('data', exist_ok=True)
with open('data/orders.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=records[0].keys())
    writer.writeheader()
    writer.writerows(records)

# Save as JSON (newline-delimited)
with open('data/orders.json', 'w') as f:
    for r in records:
        f.write(json.dumps(r) + '\n')

# Save as Parquet via pyarrow
table = pa.Table.from_pylist(records)
pq.write_table(table, 'data/orders.parquet')

print('Files created:')
for fmt in ['csv', 'json', 'parquet']:
    size = os.path.getsize(f'data/orders.{fmt}')
    print(f'  {fmt.upper()}: {size:,} bytes')

# Profile read performance
def profile_format(fmt, reader_fn):
    t0 = time.time()
    data = reader_fn()
    read_ms = round((time.time() - t0) * 1000, 1)
    size_kb = os.path.getsize(f'data/orders.{fmt}') // 1024
    return {'rows': len(data), 'size_kb': size_kb, 'read_ms': read_ms}

def read_csv():
    with open('data/orders.csv') as f:
        return list(csv.DictReader(f))

def read_json():
    with open('data/orders.json') as f:
        return [json.loads(line) for line in f]

def read_parquet():
    tbl = pq.read_table('data/orders.parquet')
    return tbl.to_pylist()

print('\nRead Performance:')
for fmt, reader in [('csv', read_csv), ('json', read_json), ('parquet', read_parquet)]:
    p = profile_format(fmt, reader)
    print(f'  {fmt.upper()}: {p["rows"]} rows | {p["size_kb"]} KB | {p["read_ms"]}ms read')

# Verify Parquet is smaller than CSV
csv_size = os.path.getsize('data/orders.csv')
pq_size = os.path.getsize('data/orders.parquet')
print(f'\nCompression: CSV={csv_size} bytes, Parquet={pq_size} bytes')
print(f'Parquet is {csv_size/pq_size:.1f}x smaller')

Files created:
  CSV: 9,746 bytes
  JSON: 18,490 bytes
  PARQUET: 4,452 bytes

Read Performance:
  CSV: 100 rows | 9 KB | 0.6ms read
  JSON: 100 rows | 18 KB | 1.1ms read
  PARQUET: 100 rows | 4 KB | 3.7ms read

Compression: CSV=9746 bytes, Parquet=4452 bytes
Parquet is 2.2x smaller


In [23]:
# Cell 5: Results Analysis
print('=' * 60)
print('RESULTS: Structured vs Semi-Structured Data')
print('=' * 60)
print(f'Input rows: {len(df)}')
print(f'Processing complete')

# Display key metrics
print(f'\nRevenue by Region:')
print(df.groupby('region')['revenue'].sum().sort_values(ascending=False))

RESULTS: Structured vs Semi-Structured Data
Input rows: 10000
Processing complete

Revenue by Region:
region
West_with_extended_description     32704052.11
East_with_extended_description     31929463.06
South_with_extended_description    30902956.65
North_with_extended_description    30756088.37
Name: revenue, dtype: float64


In [24]:
# Cell 6: Self-Check — Structured vs Semi-Structured Data
# Run this cell to verify your work before clicking "Run Tests"
print('=' * 50)
print('SELF-CHECK — Structured vs Semi-Structured Data')
print('=' * 50)

checks = [
    ('os.path.exists("data/orders.csv")', lambda: os.path.exists("data/orders.csv")),
    ('os.path.exists("data/orders.json")', lambda: os.path.exists("data/orders.json")),
    ('os.path.exists("data/orders.parquet")', lambda: os.path.exists("data/orders.parquet")),
    ('len(df) >= 100', lambda: len(df) >= 100),
    ('os.path.getsize("data/orders.parquet") < os.path.getsize("data/orders.csv")', lambda: os.path.getsize("data/orders.parquet") < os.path.getsize("data/orders.csv")),
    ('len(df.columns) >= 5', lambda: len(df.columns) >= 5),
    ('df.isnull().sum().sum() == 0', lambda: df.isnull().sum().sum() == 0),
    ('df.dtypes["quantity"] in ["int64","int32"]', lambda: df.dtypes["quantity"] in ["int64","int32"]),
]

passed = 0
for name, fn in checks:
    try:
        result = fn()
        status = 'PASS' if result else 'FAIL'
        if result: passed += 1
    except Exception as e:
        status = f'ERROR: {e}'
    print(f'  {status}: {name}')

print(f'\n{passed}/{len(checks)} self-checks passed')
if passed == len(checks):
    print('\nAll good! Click "Run Tests" to submit for official validation.')
else:
    print('\nSome checks failed. Fix your code, then click "Run Tests".')

SELF-CHECK — Structured vs Semi-Structured Data
  PASS: os.path.exists("data/orders.csv")
  PASS: os.path.exists("data/orders.json")
  PASS: os.path.exists("data/orders.parquet")
  PASS: len(df) >= 100
  PASS: os.path.getsize("data/orders.parquet") < os.path.getsize("data/orders.csv")
  PASS: len(df.columns) >= 5
  PASS: df.isnull().sum().sum() == 0
  PASS: df.dtypes["quantity"] in ["int64","int32"]

8/8 self-checks passed

All good! Click "Run Tests" to submit for official validation.
